In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

In [3]:
df = pd.read_csv('fire_nrt_J1V-C2_725729.csv')
df.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-18.99333,-174.76727,338.14,0.61,0.71,2026-01-01,26,N20,VIIRS,n,2.0NRT,284.34,12.02,D
1,-18.99204,-174.76611,342.67,0.61,0.71,2026-01-01,26,N20,VIIRS,n,2.0NRT,286.74,9.88,D
2,67.55039,83.20029,348.26,0.73,0.76,2026-01-01,58,N20,VIIRS,n,2.0NRT,249.40,8.95,N
3,71.19350,67.09715,321.12,0.61,0.53,2026-01-01,58,N20,VIIRS,n,2.0NRT,255.69,3.41,N
4,68.52409,79.94820,320.47,0.59,0.70,2026-01-01,58,N20,VIIRS,n,2.0NRT,248.18,4.18,N


In [4]:
df.isnull().sum()

latitude      0
longitude     0
brightness    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_t31    0
frp           0
daynight      0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1013206 entries, 0 to 1013205
Data columns (total 14 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   latitude    1013206 non-null  float64
 1   longitude   1013206 non-null  float64
 2   brightness  1013206 non-null  float64
 3   scan        1013206 non-null  float64
 4   track       1013206 non-null  float64
 5   acq_date    1013206 non-null  object 
 6   acq_time    1013206 non-null  int64  
 7   satellite   1013206 non-null  object 
 8   instrument  1013206 non-null  object 
 9   confidence  1013206 non-null  object 
 10  version     1013206 non-null  object 
 11  bright_t31  1013206 non-null  float64
 12  frp         1013206 non-null  float64
 13  daynight    1013206 non-null  object 
dtypes: float64(7), int64(1), object(6)
memory usage: 108.2+ MB


## Feature Transformation based on our EDA 

### Log transforming frp

In [6]:
df['frp'] = np.log1p(df['frp'])

### Date and time features

In [7]:
df['acq_date'] = pd.to_datetime(df['acq_date'])
df['month'] = df['acq_date'].dt.month

### Encoding categorical variables

In [8]:
df['confidence'] = df['confidence'].map({'l': 0, 'n': 1, 'h': 2})
df['is_day'] = (df['daynight'] == 'D').astype(int)

### Dropping redundant columns

In [9]:
df.drop(columns=['version', 'daynight', 'acq_date', 'satellite', 'instrument'], inplace=True)

In [10]:
df.head()

,latitude,longitude,brightness,scan,track,acq_time,confidence,bright_t31,frp,month,is_day
0,-18.99333,-174.76727,338.14,0.61,0.71,26,1,284.34,2.566487,1,1
1,-18.99204,-174.76611,342.67,0.61,0.71,26,1,286.74,2.386926,1,1
2,67.55039,83.20029,348.26,0.73,0.76,58,1,249.40,2.297573,1,0
3,71.19350,67.09715,321.12,0.61,0.53,58,1,255.69,1.483875,1,0
4,68.52409,79.94820,320.47,0.59,0.70,58,1,248.18,1.644805,1,0


## Splitting: 80-20 because Grid Search CV does validation splits

In [11]:
X = df.drop(columns=['frp'])
y = df['frp']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Training model

In [12]:
lr = LinearRegression()
lr.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### Test

In [13]:
y_test_pred = lr.predict(X_test)
mse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)
print(f"Linear Regression - MSE: {mse:.4f}, R²: {r2:.4f}")

Linear Regression - MSE: 0.3136, R²: 0.5216


### Grid Search CV does Cross Validation. So, we do not need a separate validation split. Combining train and val

#### Using ridge regularization for grid search

In [14]:
ridge = Ridge()

param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100]  # regularization strength
}

grid = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END .........................................alpha=0.01; total time=   0.1s
[CV] END .........................................alpha=0.01; total time=   0.2s
[CV] END ..........................................alpha=0.1; total time=   0.2s
[CV] END ..........................................alpha=0.1; total time=   0.2s
[CV] END .........................................alpha=0.01; total time=   0.3s
[CV] END ............................................alpha=1; total time=   0.3s
[CV] END ..........................................alpha=0.1; total time=   0.3s
[CV] END ...........................................alpha=10; total time=   0.3s
[CV] END ...........................................alpha=10; total time=   0.3s
[CV] END ...........................................alpha=10; total time=   0.3s
[CV] END ............................................alpha=1; total time=   0.3s
[CV] END ........................................

,estimator,Ridge()
,param_grid,"{'alpha': [0.01, 0.1, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.1


In [15]:
print("Best alpha:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

Best alpha: {'alpha': 0.1}
Best CV R²: 0.5193168757757229


### Eval on test

In [16]:
best_model = grid.best_estimator_

y_test_pred = best_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("Test Results")
print("RMSE:", test_rmse)
print("R²:", test_r2)

Test Results
RMSE: 0.5600345125487657
R²: 0.5215753170647696


## Random Forest

In [17]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X_train, y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


KeyboardInterrupt: 